In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
conn = sqlite3.connect("seguranca.db")

query = """
SELECT 
    id,
    colaborador_id,
    nome_colaborador,
    tag_rfid,
    timestamp,
    tipo_evento
FROM logs_acesso
"""

df = pd.read_sql_query(query, conn)

conn.close()

In [ ]:
df.head()

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["data"] = df["timestamp"].dt.date

In [ ]:
entradas = df[df["tipo_evento"] == "Entrada"]
saidas = df[df["tipo_evento"] == "Saída"]
negados = df[df["tipo_evento"] == "Tentativa Negada"]
invasoes = df[df["tipo_evento"] == "Tag Não Reconhecida"]

In [ ]:
print("Entradas:", len(entradas))
print("Saídas:", len(saidas))
print("Negados:", len(negados))
print("Invasões:", len(invasoes))

In [ ]:
negados["nome_colaborador"].value_counts()

In [ ]:
movimentacoes = df[
    (df["tipo_evento"] == "Entrada") |
    (df["tipo_evento"] == "Saída")
]

usuarios = movimentacoes["nome_colaborador"].unique()

resultado_tempos = []

for usuario in usuarios:

    dados_usuario = movimentacoes[
        movimentacoes["nome_colaborador"] == usuario
    ]

    entradas_usuario = dados_usuario[
        dados_usuario["tipo_evento"] == "Entrada"
    ]["timestamp"].reset_index(drop=True)

    saidas_usuario = dados_usuario[
        dados_usuario["tipo_evento"] == "Saída"
    ]["timestamp"].reset_index(drop=True)

    quantidade = min(len(entradas_usuario), len(saidas_usuario))

    tempo_total = pd.Timedelta(0)

    for i in range(quantidade):
        tempo_total += (saidas_usuario[i] - entradas_usuario[i])

    resultado_tempos.append({
        "usuario": usuario,
        "tempo_total": tempo_total
    })

df_tempo = pd.DataFrame(resultado_tempos)
df_tempo

In [ ]:
df_tempo["minutos"] = df_tempo["tempo_total"].dt.total_seconds() / 60
df_tempo

In [ ]:
status = entradas["nome_colaborador"].value_counts().subtract(
    saidas["nome_colaborador"].value_counts(),
    fill_value=0
)

status[status > 0]

In [ ]:
eventos = df["tipo_evento"].value_counts()
eventos

In [ ]:
top5 = df_tempo.sort_values("minutos", ascending=False).head(5)

fig, axs = plt.subplots(3, 1, figsize=(10, 14))

# =========================
# Gráfico 1 - Tempo na sala
# =========================

df_tempo.plot(
    x="usuario",
    y="minutos",
    kind="bar",
    legend=False,
    ax=axs[0]
)

axs[0].set_title("Tempo de Permanência na Sala")
axs[0].set_xlabel("Usuário")
axs[0].set_ylabel("Minutos")

# =========================
# Gráfico 2 - Eventos
# =========================

eventos.plot(
    kind="bar",
    ax=axs[1]
)

axs[1].set_title("Eventos do Sistema")
axs[1].set_xlabel("Tipo de Evento")
axs[1].set_ylabel("Quantidade")

# =========================
# Gráfico 3 - TOP 5 tempo
# =========================

top5.plot(
    x="usuario",
    y="minutos",
    kind="bar",
    legend=False,
    ax=axs[2]
)

axs[2].set_title("Top 5 Usuários por Tempo")
axs[2].set_xlabel("Usuário")
axs[2].set_ylabel("Minutos")

plt.tight_layout()
plt.show()

In [ ]:
df.to_csv("relatorio_logs.csv", index=False)

print("Relatório exportado com sucesso!")